# SQ26 — Part 2: Classification
**Student:** Shubhangi More | **ID:** 23137504  
**Repositories:** Zenodo (id=1) · Harvard Dataverse (id=10)

### Notebook flow
| # | Section | Output |
|---|---|---|
| 1 | Setup & paths | — |
| 2 | Connect to acquisition DB | — |
| 3 | Pre-flight: folder check, DB reconcile, HTML check | Fixed `files` table |
| 4 | Build classification DB | `23137504-sq26-classification.db` |
| 5 | Assign PROJECT_TYPE | `type` column |
| 6 | Load ISIC taxonomy | In-memory dict |
| 7 | Load model + **sanity gate** (blocks if model broken) | Validated embeddings |
| 8 | Run classifier | `primary_class`, `secondary_class`, `similarity_score` |
| 9 | Export XLSX | `23137504-sq26-classification-results.xlsx` |
| 10 | Print statistics | Console |
| 11 | Generate PDF report | `23137504-sq26-classification-report.pdf` |
| 12 | Final summary + checklist | — |


## 1 — Setup & Imports

In [96]:
import sqlite3
import re
import textwrap
import sys
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.backends.backend_pdf import PdfPages
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print(f'Python : {sys.version}')
print('Imports OK')


Python : 3.9.6 (default, Apr 17 2026, 18:15:52) 
[Clang 21.0.0 (clang-2100.1.1.101)]
Imports OK


## 2 — Paths & Constants

In [97]:
STUDENT_ID   = "23137504"
STUDENT_NAME = "Shubhangi More"

SEEDING_DB   = Path(f"{STUDENT_ID}-sq26.db")
CLASS_DB     = Path(f"{STUDENT_ID}-sq26-classification.db")
XLSX_OUT     = Path(f"{STUDENT_ID}-sq26-classification-results.xlsx")
PDF_OUT      = Path(f"{STUDENT_ID}-sq26-classification-report.pdf")  # kept for reference / single-file submission if needed
def pdf_out_for(repo_name):
    """Per-repository PDF path, e.g. 23137504-sq26-classification-report-zenodo.pdf"""
    slug = repo_name.lower().replace(" ", "-")
    return Path(f"{STUDENT_ID}-sq26-classification-report-{slug}.pdf")
DOWNLOAD_ROOT = Path("downloads")

REPO_NAMES = {1: 'Zenodo', 10: 'Harvard Dataverse'}

QDA_EXTENSIONS = {
    "qdpx",
    "mx24","mx22","mx20","mx18","mx14","mx12","mx11","mqd","mqdc1",
    "nvp","nvpx","nvb",
    "atlproj","atlasproj","hpr7","hpr6",
    "ppj","pprj","f4p","f4a","quirkos","qde",
}
PRIMARY_EXTENSIONS = {
    "pdf","txt","rtf","doc","docx","odt",
    "xlsx","xls","ods","csv","tsv","tab",
    "html","htm","xml",
    "mp3","mp4","wav","m4a","ogg","mov",
    "jpg","jpeg","png","tif","tiff",
}

print(f"Seeding DB   : {SEEDING_DB}")
print(f"Class DB     : {CLASS_DB}")
print(f"Download root: {DOWNLOAD_ROOT.resolve()}")


Seeding DB   : 23137504-sq26.db
Class DB     : 23137504-sq26-classification.db
Download root: /Users/shubhangimore/seeding_qa_analytics_project/downloads


## 3 — Connect to Acquisition DB

In [98]:
if not SEEDING_DB.exists():
    raise FileNotFoundError(
        f"Cannot find '{SEEDING_DB}'.\n"
        "Run the acquisition notebook first and make sure you're in the same folder."
    )

src = sqlite3.connect(SEEDING_DB)
src.row_factory = sqlite3.Row

tables = [r[0] for r in src.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall()]
print("Tables:", tables)
n_proj  = src.execute('SELECT COUNT(*) FROM projects').fetchone()[0]
n_files = src.execute('SELECT COUNT(*) FROM files').fetchone()[0]
print(f"Projects in seeding DB : {n_proj}")
print(f"File rows in seeding DB: {n_files}")


Tables: ['repositories', 'projects', 'sqlite_sequence', 'files', 'keywords', 'person_role', 'licenses']
Projects in seeding DB : 363
File rows in seeding DB: 11835


## 4 — Pre-Flight Diagnostics

Four checks before touching the classification DB:
1. Download folder exists on disk
2. Reconcile: projects with files on disk but missing `SUCCEEDED` rows → backfill
3. Description HTML check + define `strip_html()` used by the classifier
4. Scope preview — how many projects will be classified


In [99]:
# ── 4.1  Download folder ────────────────────────────────────────────────────
print('=== 4.1 Download folder check ===')
if not DOWNLOAD_ROOT.exists():
    print(f'  WARNING: {DOWNLOAD_ROOT.resolve()} does not exist.')
    print('  Run this notebook from the same directory as the acquisition notebook.')
else:
    sub = [p.name for p in DOWNLOAD_ROOT.iterdir() if p.is_dir()]
    print(f"  Path     : {DOWNLOAD_ROOT.resolve()}")
    print(f"  Sub-dirs : {sub}")
    total_on_disk = sum(1 for _ in DOWNLOAD_ROOT.rglob('*') if _.is_file())
    print(f"  Files on disk: {total_on_disk}")


=== 4.1 Download folder check ===
  Path     : /Users/shubhangimore/seeding_qa_analytics_project/downloads
  Sub-dirs : ['harvard-dataverse', 'zenodo']
  Files on disk: 3738


In [100]:
# ── 4.2  Reconcile DB vs disk ───────────────────────────────────────────────
# Projects that were scraped with dry_run=True have metadata but no file rows.
# If the files were later downloaded manually, we backfill the DB here.
print('=== 4.2 Reconcile DB vs disk ===')

no_files_rows = src.execute("""
    SELECT p.id, p.download_repository_folder, p.download_project_folder
    FROM projects p
    WHERE NOT EXISTS (
        SELECT 1 FROM files f
        WHERE f.project_id = p.id AND f.status = 'SUCCEEDED'
    )
""").fetchall()

print(f"  Projects with 0 SUCCEEDED file rows : {len(no_files_rows)}")

backfilled_p = backfilled_f = 0
for proj in no_files_rows:
    disk_dir = DOWNLOAD_ROOT / proj['download_repository_folder'] / proj['download_project_folder']
    if not disk_dir.exists():
        continue
    disk_files = [f for f in disk_dir.rglob('*') if f.is_file()]
    if not disk_files:
        continue
    backfilled_p += 1
    for df in disk_files:
        fname = df.name
        ext   = fname.rsplit('.', 1)[-1].lower() if '.' in fname else 'unknown'
        exists = src.execute(
            'SELECT 1 FROM files WHERE project_id=? AND file_name=?',
            (proj['id'], fname)
        ).fetchone()
        if not exists:
            src.execute(
                'INSERT INTO files (project_id,file_name,file_type,status) VALUES(?,?,?,"SUCCEEDED")',
                (proj['id'], fname, ext)
            )
            backfilled_f += 1
src.commit()

print(f"  Backfilled: {backfilled_p} projects, {backfilled_f} file rows")
still = src.execute("""
    SELECT COUNT(*) FROM projects p WHERE NOT EXISTS(
        SELECT 1 FROM files f WHERE f.project_id=p.id AND f.status='SUCCEEDED')
""").fetchone()[0]
print(f"  Still 0-file after reconcile: {still}  (genuinely not on disk)")


=== 4.2 Reconcile DB vs disk ===
  Projects with 0 SUCCEEDED file rows : 115
  Backfilled: 0 projects, 0 file rows
  Still 0-file after reconcile: 115  (genuinely not on disk)


In [101]:
# ── 4.3  HTML stripper + description quality ────────────────────────────────
def strip_html(text: str) -> str:
    """Strip HTML tags and decode entities. Critical for clean embeddings."""
    if not text:
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)
    for ent, ch in [
        ('&amp;','&'),('&lt;','<'),('&gt;','>'),
        ('&quot;','"'),('&#39;',"'"),('&nbsp;',' '),
        ('&ndash;','-'),('&mdash;','-'),('&hellip;','...'),
    ]:
        text = text.replace(ent, ch)
    return re.sub(r'\s+', ' ', text).strip()

print('=== 4.3 Description quality ===')
total  = src.execute('SELECT COUNT(*) FROM projects').fetchone()[0]
html_n = sum(1 for r in src.execute('SELECT description FROM projects').fetchall()
             if r['description'] and '<' in r['description'])
empty_n = sum(1 for r in src.execute('SELECT description FROM projects').fetchall()
              if not (r['description'] or '').strip())
print(f"  Total projects         : {total}")
print(f"  Descriptions with HTML : {html_n}  ({html_n/total*100:.0f}%)")
print(f"  Empty descriptions     : {empty_n}")

# Show 2 before/after examples
samples = src.execute(
    "SELECT id, title, description FROM projects WHERE description LIKE '<%%' LIMIT 2"
).fetchall()
if samples:
    print("\n  Before/after HTML strip:")
    for r in samples:
        print(f"  [{r['id']}] {r['title'][:55]}")
        print(f"    RAW  : {(r['description'] or '')[:90]}")
        print(f"    CLEAN: {strip_html(r['description'] or '')[:90]}")
        print()


=== 4.3 Description quality ===
  Total projects         : 363
  Descriptions with HTML : 36  (10%)
  Empty descriptions     : 0

  Before/after HTML strip:
  [1] Supporting Data for Scoping Review on Interlanguage Pra
    RAW  : <p>This dataset contains coded data (QDPX), article summary (Word), and year-theme matrix 
    CLEAN: This dataset contains coded data (QDPX), article summary (Word), and year-theme matrix (Ex

  [2] Johnson Pandemic Corpus 2020 (JPanC20)
    RAW  : <h1>The Johnson Pandemic Corpus 2020 (JPanC20)</h1>
<h2>Pragmatically annotated corpus of 
    CLEAN: The Johnson Pandemic Corpus 2020 (JPanC20) Pragmatically annotated corpus of political spe



In [102]:
# ── 4.4  Scope preview ───────────────────────────────────────────────────────
print('=== 4.4 Classification scope preview ===')
preview = Counter()
for proj in src.execute('SELECT id FROM projects').fetchall():
    pid  = proj['id']
    exts = {
        r['file_type'].lower().strip('.')
        for r in src.execute(
            'SELECT file_type FROM files WHERE project_id=? AND status="SUCCEEDED"', (pid,)
        ).fetchall() if r['file_type']
    }
    if exts & QDA_EXTENSIONS:         preview['QDA_PROJECT'] += 1
    elif exts & PRIMARY_EXTENSIONS:   preview['QD_PROJECT'] += 1
    elif exts:                         preview['OTHER_PROJECT'] += 1
    else:                              preview['NOT_A_PROJECT'] += 1

eligible = preview['QDA_PROJECT'] + preview['QD_PROJECT']
for t in ['QDA_PROJECT','QD_PROJECT','OTHER_PROJECT','NOT_A_PROJECT']:
    tag = ' ← will be classified' if t in ('QDA_PROJECT','QD_PROJECT') else ''
    print(f"  {t:<20} {preview[t]:>5}{tag}")
print(f"\n  Total eligible: {eligible}")


=== 4.4 Classification scope preview ===
  QDA_PROJECT             31 ← will be classified
  QD_PROJECT             187 ← will be classified
  OTHER_PROJECT           30
  NOT_A_PROJECT          115

  Total eligible: 218


## 5 — Create Classification DB

In [103]:
def create_classification_db(path: Path) -> sqlite3.Connection:
    if path.exists():
        path.unlink()
    con = sqlite3.connect(path)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS projects (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            query_string TEXT, repository_id INTEGER NOT NULL,
            repository_url TEXT NOT NULL, project_url TEXT NOT NULL UNIQUE,
            version TEXT, title TEXT NOT NULL, description TEXT NOT NULL,
            language TEXT, doi TEXT, upload_date DATE,
            download_date TIMESTAMP NOT NULL,
            download_repository_folder TEXT NOT NULL,
            download_project_folder TEXT NOT NULL,
            download_version_folder TEXT, download_method TEXT NOT NULL,
            type TEXT CHECK(type IN
                ('QDA_PROJECT','QD_PROJECT','OTHER_PROJECT','NOT_A_PROJECT')),
            primary_class TEXT, secondary_class TEXT,
            classified_at TEXT, similarity_score REAL, has_qda_files INTEGER
        )""")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS files (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            project_id INTEGER NOT NULL REFERENCES projects(id),
            file_name TEXT NOT NULL, file_type TEXT NOT NULL, status TEXT NOT NULL
        )""")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS file_classification (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            project_id INTEGER NOT NULL REFERENCES projects(id),
            file_name TEXT NOT NULL,
            primary_class TEXT, secondary_class TEXT, similarity_score REAL
        )""")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS keywords (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            project_id INTEGER NOT NULL REFERENCES projects(id),
            keyword TEXT NOT NULL)""")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS person_role (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            project_id INTEGER NOT NULL REFERENCES projects(id),
            name TEXT NOT NULL, role TEXT NOT NULL)""")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS licenses (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            project_id INTEGER NOT NULL REFERENCES projects(id),
            license TEXT NOT NULL)""")
    con.commit()
    print(f"Classification DB created: {path.resolve()}")
    return con

dst = create_classification_db(CLASS_DB)


Classification DB created: /Users/shubhangimore/seeding_qa_analytics_project/23137504-sq26-classification.db


## 6 — Copy Acquisition Data

In [104]:
def copy_data(src, dst):
    cur = dst.cursor()
    id_map = {}
    for row in src.execute('SELECT * FROM projects').fetchall():
        r = dict(row); old_id = r.pop('id')
        cur.execute("""
            INSERT OR IGNORE INTO projects(
                query_string,repository_id,repository_url,project_url,version,
                title,description,language,doi,upload_date,download_date,
                download_repository_folder,download_project_folder,
                download_version_folder,download_method)
            VALUES(:query_string,:repository_id,:repository_url,:project_url,:version,
                :title,:description,:language,:doi,:upload_date,:download_date,
                :download_repository_folder,:download_project_folder,
                :download_version_folder,:download_method)""", r)
        new_id = cur.lastrowid or dst.execute(
            'SELECT id FROM projects WHERE project_url=?',(r['project_url'],)
        ).fetchone()['id']
        id_map[old_id] = new_id

    fmap = {}
    for row in src.execute('SELECT * FROM files').fetchall():
        r = dict(row); old_fid = r.pop('id')
        new_pid = id_map.get(r['project_id'])
        if not new_pid: continue
        r['project_id'] = new_pid
        cur.execute('INSERT OR IGNORE INTO files(project_id,file_name,file_type,status)'
                    ' VALUES(:project_id,:file_name,:file_type,:status)', r)
        if cur.lastrowid: fmap[old_fid] = cur.lastrowid

    for row in src.execute('SELECT * FROM keywords').fetchall():
        r = dict(row); r.pop('id')
        new_pid = id_map.get(r['project_id'])
        if new_pid: cur.execute('INSERT INTO keywords(project_id,keyword) VALUES(?,?)',
                                (new_pid, r['keyword']))
    for row in src.execute('SELECT * FROM person_role').fetchall():
        r = dict(row); r.pop('id')
        new_pid = id_map.get(r['project_id'])
        if new_pid: cur.execute('INSERT INTO person_role(project_id,name,role) VALUES(?,?,?)',
                                (new_pid, r['name'], r['role']))
    for row in src.execute('SELECT * FROM licenses').fetchall():
        r = dict(row); r.pop('id')
        new_pid = id_map.get(r['project_id'])
        if new_pid: cur.execute('INSERT INTO licenses(project_id,license) VALUES(?,?)',
                                (new_pid, r['license']))
    dst.commit()
    print(f"Copied: {len(id_map)} projects, {len(fmap)} files")
    return id_map

id_map = copy_data(src, dst)


Copied: 363 projects, 5305 files


## 7 — Assign PROJECT_TYPE

In [105]:
def assign_project_type(dst):
    counts = Counter()
    for proj in dst.execute('SELECT id FROM projects').fetchall():
        pid  = proj['id']
        exts = {
            r['file_type'].lower().strip('.')
            for r in dst.execute(
                'SELECT file_type FROM files WHERE project_id=? AND status="SUCCEEDED"',(pid,)
            ).fetchall() if r['file_type']
        }
        has_qda = bool(exts & QDA_EXTENSIONS)
        if has_qda:                       pt = 'QDA_PROJECT'
        elif exts & PRIMARY_EXTENSIONS:   pt = 'QD_PROJECT'
        elif exts:                         pt = 'OTHER_PROJECT'
        else:                              pt = 'NOT_A_PROJECT'
        dst.execute('UPDATE projects SET type=?,has_qda_files=? WHERE id=?',
                    (pt, int(has_qda), pid))
        counts[pt] += 1
    dst.commit()
    print("PROJECT_TYPE distribution:")
    for t in ['QDA_PROJECT','QD_PROJECT','OTHER_PROJECT','NOT_A_PROJECT']:
        print(f"  {t:<20} {counts.get(t,0):>5}")
    return counts

type_counts = assign_project_type(dst)


PROJECT_TYPE distribution:
  QDA_PROJECT             31
  QD_PROJECT             187
  OTHER_PROJECT           30
  NOT_A_PROJECT          115


## 8 — Full ISIC Rev.5 Taxonomy

In [106]:
ISIC = {
    "01":"Crop and animal production, hunting and related service activities",
    "02":"Forestry and logging","03":"Fishing and aquaculture",
    "05":"Mining of coal and lignite","06":"Extraction of crude petroleum and natural gas",
    "07":"Mining of metal ores","08":"Other mining and quarrying",
    "09":"Mining support service activities","10":"Manufacture of food products",
    "11":"Manufacture of beverages","12":"Manufacture of tobacco products",
    "13":"Manufacture of textiles","14":"Manufacture of wearing apparel",
    "15":"Manufacture of leather and related products",
    "16":"Manufacture of wood and wood products",
    "17":"Manufacture of paper and paper products",
    "18":"Printing and reproduction of recorded media",
    "19":"Manufacture of coke and refined petroleum products",
    "20":"Manufacture of chemicals and chemical products",
    "21":"Manufacture of basic pharmaceutical products and pharmaceutical preparations",
    "22":"Manufacture of rubber and plastic products",
    "23":"Manufacture of other non-metallic mineral products",
    "24":"Manufacture of basic metals",
    "25":"Manufacture of fabricated metal products, except machinery and equipment",
    "26":"Manufacture of computer, electronic and optical products",
    "27":"Manufacture of electrical equipment",
    "28":"Manufacture of machinery and equipment not elsewhere classified",
    "29":"Manufacture of motor vehicles, trailers and semi-trailers",
    "30":"Manufacture of other transport equipment","31":"Manufacture of furniture",
    "32":"Other manufacturing","33":"Repair and installation of machinery and equipment",
    "35":"Electricity, gas, steam and air conditioning supply",
    "36":"Water collection, treatment and supply","37":"Sewerage",
    "38":"Waste collection, treatment and disposal activities",
    "39":"Remediation activities and other waste management services",
    "41":"Construction of buildings","42":"Civil engineering",
    "43":"Specialised construction activities",
    "45":"Wholesale and retail trade and repair of motor vehicles and motorcycles",
    "46":"Wholesale trade, except of motor vehicles and motorcycles",
    "47":"Retail trade, except of motor vehicles and motorcycles",
    "49":"Land transport and transport via pipelines","50":"Water transport",
    "51":"Air transport","52":"Warehousing and support activities for transportation",
    "53":"Postal and courier activities","55":"Accommodation",
    "56":"Food and beverage service activities","58":"Publishing activities",
    "59":"Motion picture, video and television programme production, sound recording and music publishing activities",
    "60":"Programming and broadcasting activities","61":"Telecommunications",
    "62":"Computer programming, consultancy and related activities",
    "63":"Information service activities",
    "64":"Financial service activities, except insurance and pension funding",
    "65":"Insurance, reinsurance and pension funding, except compulsory social security",
    "66":"Activities auxiliary to financial service and insurance activities",
    "68":"Real estate activities","69":"Legal and accounting activities",
    "70":"Activities of head offices; management consultancy activities",
    "71":"Architectural and engineering activities; technical testing and analysis",
    "72":"Scientific research and development","73":"Advertising and market research",
    "74":"Other professional, scientific and technical activities",
    "75":"Veterinary activities","77":"Rental and leasing activities",
    "78":"Employment activities",
    "79":"Travel agency, tour operator and other reservation service and related activities",
    "80":"Security and investigation activities",
    "81":"Services to buildings and landscape activities",
    "82":"Office administrative, office support and other business support activities",
    "84":"Public administration and defence; compulsory social security",
    "85":"Education","86":"Human health activities","87":"Residential care activities",
    "88":"Social work activities without accommodation",
    "90":"Creative, arts and entertainment activities",
    "91":"Libraries, archives, museums and other cultural activities",
    "92":"Gambling and betting activities",
    "93":"Sports activities and amusement and recreation activities",
    "94":"Activities of membership organisations",
    "95":"Repair of computers and personal and household goods",
    "96":"Other personal service activities",
    "97":"Activities of households as employers of domestic personnel",
    "98":"Undifferentiated goods- and services-producing activities of private households for own use",
    "99":"Activities of extraterritorial organisations and bodies",
}
ISIC_CODES  = list(ISIC.keys())

# ── FIX: keyword-enriched labels ─────────────────────────────────────────────
# The bare ISIC label (e.g. "Human health activities") is 3-6 words. A project
# description is 100-2000+ words. Cosine similarity between a long, concrete
# text and a short, generic label is structurally weak - not because the
# model is broken, but because there's little vocabulary for it to match on.
# Appending a small set of concrete, dataset-relevant keywords per division
# gives the embedding model more surface area to match against.
ISIC_KEYWORDS = {
    "01":"farming, agriculture, livestock, crops, hunting, farmers",
    "02":"forestry, logging, timber, forest management",
    "03":"fishing, aquaculture, fisheries, fish farming",
    "05":"coal mining, lignite, coal extraction",
    "06":"oil, gas, petroleum extraction, drilling",
    "07":"metal ore mining, mining minerals",
    "08":"quarrying, mining, stone, sand, gravel",
    "09":"mining support, drilling services",
    "10":"food processing, food manufacturing, food products",
    "11":"beverages, brewing, drinks manufacturing",
    "12":"tobacco, cigarettes, tobacco products",
    "13":"textiles, fabric, weaving, spinning",
    "14":"clothing, apparel, garments, fashion manufacturing",
    "15":"leather, footwear, leather goods",
    "16":"wood products, timber processing, carpentry",
    "17":"paper, pulp, paper products",
    "18":"printing, publishing production, recorded media reproduction",
    "19":"petroleum refining, coke, refined oil products",
    "20":"chemicals, chemical industry, chemical products",
    "21":"pharmaceuticals, drug manufacturing, medicines production",
    "22":"rubber, plastics, plastic products",
    "23":"cement, glass, ceramics, non-metallic minerals",
    "24":"steel, iron, basic metals, metal production",
    "25":"metal fabrication, metalwork, fabricated metal products",
    "26":"computers, electronics, optical instruments manufacturing",
    "27":"electrical equipment, appliances manufacturing",
    "28":"machinery manufacturing, industrial equipment",
    "29":"automobiles, cars, motor vehicle manufacturing",
    "30":"ships, trains, aircraft manufacturing, transport equipment",
    "31":"furniture manufacturing, furniture design",
    "32":"other manufacturing, misc manufacturing, toys, jewellery",
    "33":"machinery repair, equipment maintenance, installation services",
    "35":"electricity, energy, power supply, gas, utilities",
    "36":"water supply, drinking water, water treatment",
    "37":"sewerage, wastewater, sewage treatment",
    "38":"waste management, recycling, waste collection, landfill",
    "39":"environmental remediation, pollution cleanup",
    "41":"construction, building, real estate development, housing construction",
    "42":"civil engineering, infrastructure, roads, bridges, construction projects",
    "43":"specialised construction, electrical installation, plumbing, demolition",
    "45":"motor vehicle sales, car dealers, vehicle repair",
    "46":"wholesale trade, distribution, wholesalers",
    "47":"retail trade, shops, stores, retailers",
    "49":"land transport, railways, trucking, road transport, buses",
    "50":"water transport, shipping, maritime, ferries",
    "51":"air transport, aviation, airlines, aircraft, flights",
    "52":"warehousing, logistics, cargo handling, transport support",
    "53":"postal service, courier, mail delivery, parcel delivery",
    "55":"hotels, accommodation, hospitality, lodging, guesthouses, tourism",
    "56":"restaurants, catering, food service, cafes, bars",
    "58":"publishing, books, newspapers, journals, magazines",
    "59":"film, movies, video production, television programmes, music recording",
    "60":"broadcasting, radio, television programming, streaming",
    "61":"telecommunications, mobile networks, internet providers, telephony",
    "62":"software development, programming, coding, IT consultancy, computer science",
    "63":"information services, data processing, web portals, search engines, hosting",
    "64":"banking, finance, financial services, investment",
    "65":"insurance, pension funds, reinsurance",
    "66":"financial auxiliary services, brokerage, fund management support",
    "68":"real estate, property, housing market",
    "69":"legal services, law, accounting, auditing, tax advisory",
    "70":"management consultancy, corporate strategy, business administration head offices",
    "71":"architecture, engineering, technical testing, structural analysis, urban planning",
    "72":"scientific research, R&D, academic research, laboratory research, experiments",
    "73":"advertising, marketing, market research, public relations",
    "74":"professional and technical services, design, translation, photography",
    "75":"veterinary care, animal health, vets",
    "77":"rental, leasing, equipment hire",
    "78":"employment agencies, recruitment, staffing, job placement",
    "79":"travel agency, tourism, tour operators, booking services",
    "80":"security services, investigation, surveillance, private security",
    "81":"cleaning services, landscaping, building maintenance, groundskeeping",
    "82":"office support, administrative services, call centres, business support",
    "84":"government, public administration, defence, military, policy, regulation",
    "85":"school, university, education, teaching, students, curriculum, learning, training",
    "86":"hospital, healthcare, medicine, clinical, doctors, nurses, patients, treatment",
    "87":"nursing homes, elderly care, residential care, care facilities",
    "88":"social work, community support, welfare services, counselling",
    "90":"arts, creative activities, entertainment, performing arts, artists",
    "91":"libraries, archives, museums, heritage, cultural institutions",
    "92":"gambling, betting, casinos, lottery",
    "93":"sports, recreation, fitness, amusement, leisure activities",
    "94":"membership organisations, associations, unions, NGOs, advocacy groups",
    "95":"repair services, computer repair, household goods repair",
    "96":"personal services, hairdressing, laundry, other personal care",
    "97":"domestic workers, household employers, private domestic staff",
    "98":"subsistence households, private household production",
    "99":"international organisations, extraterritorial bodies, embassies",
}
ISIC_LABELS = [f"{ISIC[c]}. Keywords: {ISIC_KEYWORDS[c]}." for c in ISIC_CODES]
print(f"ISIC Rev.5: {len(ISIC)} divisions loaded (labels enriched with keywords).")


ISIC Rev.5: 88 divisions loaded (labels enriched with keywords).


## 9 — Load Model + Sanity Gate

**Why the sanity gate matters:** On Python 3.9 + LibreSSL (macOS default),
`all-MiniLM-L6-v2` can produce near-random scores (~0.2) instead of meaningful
semantic similarity (~0.75+). The gate tests two known pairs and **raises an error
with a clear fix** if scores look wrong — so you never silently get bad classifications.

If the gate fails, change `MODEL_NAME` to `'paraphrase-MiniLM-L3-v2'` and re-run.


In [107]:
# Primary model — works on most setups
MODEL_NAME = 'paraphrase-MiniLM-L3-v2'
# NOTE: L3 was chosen to dodge a LibreSSL/Python 3.9 bug on macOS with L6, not
# because L3 gives better quality — it's the smaller, weaker of the two.
# If you are NOT on that broken macOS/Python combo, try 'all-MiniLM-L6-v2' or
# 'paraphrase-multilingual-MiniLM-L12-v2' (better if you have non-English
# descriptions - see the language check in section 4.3) and compare avg
# similarity_score at the end of the run before deciding which to keep.

print(f"Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print("Model loaded. Embedding ISIC labels...")
isic_embeddings = model.encode(ISIC_LABELS, normalize_embeddings=True,
                               show_progress_bar=True)
print(f"Embedding shape: {isic_embeddings.shape}")

# ── Sanity gate, part 1: synthetic keyword probes ───────────────────────────
# This only proves the model isn't fundamentally broken (near-random output).
# It does NOT prove real classification quality — the queries below are
# hand-built with the exact label vocabulary, so they are guaranteed to score
# well regardless of whether the pipeline works on messy real text.
SANITY_TESTS = [
    ('human health activities hospital clinic nurse doctor patient', '86', 'Human health activities'),
    ('software programming code developer machine learning',          '62', 'Computer programming'),
    ('school education teacher student curriculum learning',          '85', 'Education'),
]

print("\n=== Model sanity gate (1/2): synthetic probes ===")
gate_pass = True
for query, expected_code, expected_label in SANITY_TESTS:
    emb   = model.encode([query], normalize_embeddings=True)
    sims  = cosine_similarity(emb, isic_embeddings)[0]
    ranks = np.argsort(sims)[::-1]
    top_code  = ISIC_CODES[ranks[0]]
    top_score = float(sims[ranks[0]])
    expected_score = float(sims[ISIC_CODES.index(expected_code)])
    ok = top_score >= 0.25 and expected_score >= 0.20
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] query: '{query[:45]}'")
    print(f"         top hit : [{top_code}] {ISIC.get(top_code,'')}  score={top_score:.3f}")
    print(f"         expected: [{expected_code}] {expected_label}         score={expected_score:.3f}")
    if not ok:
        gate_pass = False

if not gate_pass:
    raise RuntimeError(
        "\n\nModel sanity gate FAILED — scores are too low for meaningful classification."
        "\nFix: change MODEL_NAME above to 'paraphrase-MiniLM-L3-v2' and re-run."
    )

# ── Sanity gate, part 2: real project text (informational, non-blocking) ────
# These are actual titles pulled from YOUR classification DB with a manually
# verified expected class. This is the test that actually tells you whether
# the pipeline works on your data - not on hand-picked keyword strings.
# It does not raise, because a low score here is diagnostic, not fatal: you
# need the run to complete so you can compare avg similarity_score against
# the reference range printed at the end of the notebook.
REAL_SANITY_TESTS = [
    ("Catalogue of Hospitals' adaptations during the Covid-pandemic", '86', 'Human health activities'),
    ('Data from: Co-creation in fully remote software teams',         '62', 'Computer programming'),
    ("A Study on the Relationship among the Principals' Space Leadership", '85', 'Education'),
]
print("\n=== Model sanity gate (2/2): real project titles from your DB ===")
for query, expected_code, expected_label in REAL_SANITY_TESTS:
    emb   = model.encode([query], normalize_embeddings=True)
    sims  = cosine_similarity(emb, isic_embeddings)[0]
    ranks = np.argsort(sims)[::-1]
    top_code  = ISIC_CODES[ranks[0]]
    top_score = float(sims[ranks[0]])
    match = "OK " if top_code == expected_code else "MISS"
    print(f"  [{match}] '{query[:55]}'")
    print(f"         predicted: [{top_code}] {ISIC.get(top_code,'')}  score={top_score:.3f}")
    print(f"         expected : [{expected_code}] {expected_label}")
print(
    "\nIf MISS shows up here on real titles, the synthetic gate above is not"
    "\ncatching your actual failure mode. Don't treat a PASS above as proof"
    "\nthe pipeline works - check this section too."
)


Loading model: paraphrase-MiniLM-L3-v2
Model loaded. Embedding ISIC labels...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding shape: (88, 384)

=== Model sanity gate (1/2): synthetic probes ===
  [PASS] query: 'human health activities hospital clinic nurse'
         top hit : [86] Human health activities  score=0.672
         expected: [86] Human health activities         score=0.672
  [PASS] query: 'software programming code developer machine l'
         top hit : [62] Computer programming, consultancy and related activities  score=0.562
         expected: [62] Computer programming         score=0.562
  [PASS] query: 'school education teacher student curriculum l'
         top hit : [85] Education  score=0.651
         expected: [85] Education         score=0.651

=== Model sanity gate (2/2): real project titles from your DB ===
  [OK ] 'Catalogue of Hospitals' adaptations during the Covid-pa'
         predicted: [86] Human health activities  score=0.240
         expected : [86] Human health activities
  [OK ] 'Data from: Co-creation in fully remote software teams'
         predicted: [62] Computer

## 10 — Classifier Function

In [108]:
def classify(text: str):
    """Embed text → cosine sim against ISIC labels → top-2 division."""
    if not text or not text.strip():
        return None, None, 0.0
    emb   = model.encode([text], normalize_embeddings=True)
    sims  = cosine_similarity(emb, isic_embeddings)[0]
    ranks = np.argsort(sims)[::-1]
    return ISIC_CODES[ranks[0]], ISIC_CODES[ranks[1]], float(sims[ranks[0]])


def get_keywords(con, pid: int) -> str:
    rows = con.execute('SELECT keyword FROM keywords WHERE project_id=?',(pid,)).fetchall()
    return ' '.join(r['keyword'] for r in rows)


## 11 — Run Classifier (Step 3)

Classifies only `QDA_PROJECT` and `QD_PROJECT`.
Per project: title×2 + stripped description + keywords → primary + secondary ISIC code.
Per primary file: filename stem + keywords → ISIC code stored in `file_classification`.


In [109]:
def build_classify_text(title: str, desc: str, kws: str) -> str:
    """
    Build the text to embed for a project.

    FIX: the original 30-word description cap was throwing away most of the
    signal - median description length in this dataset is ~216 words, so a
    30-word window kept under 15% of the description on a typical project,
    and that window is often abstract-style boilerplate (e.g. "ABSTRACT
    Objective To determine whether..."), not necessarily the most
    topic-distinctive part. Checked a handful of the lowest-scoring
    projects and several were clearly mislabeled (an HIV drug-trial dataset
    landed in "Programming and broadcasting" because "NVP" - the drug -
    collided with "NVivo" the QDA tool).

    The model truncates internally at its own max_seq_length anyway, so
    passing more text costs nothing but slightly slower encoding - there is
    no accuracy reason to pre-truncate this hard. Widened to 120 words.

    Strategy:
    - Title x2 (most reliable signal, always present)
    - First 120 words of cleaned description
    - Up to 20 keywords (add domain signal)
    """
    clean_desc = strip_html(desc or '')
    desc_words = clean_desc.split()[:120]
    desc_short = ' '.join(desc_words)
    kw_words = kws.split()[:20]
    kw_short = ' '.join(kw_words)
    return f'{title} {title} {desc_short} {kw_short}'.strip()


def run_classification(dst):
    eligible = dst.execute(
        "SELECT id,title,description FROM projects "
        "WHERE type IN ('QDA_PROJECT','QD_PROJECT')"
    ).fetchall()
    print(f"Classifying {len(eligible)} projects...")
    now = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

    low_confidence = []
    for i, proj in enumerate(eligible, 1):
        pid   = proj['id']
        title = proj['title'] or ''
        desc  = proj['description'] or ''
        kws   = get_keywords(dst, pid)

        text = build_classify_text(title, desc, kws)
        p_code, s_code, score = classify(text)

        if score is not None and score < 0.20:
            low_confidence.append((pid, title[:60], score))

        dst.execute(
            'UPDATE projects SET primary_class=?,secondary_class=?,'
            'similarity_score=?,classified_at=? WHERE id=?',
            (p_code, s_code, score, now, pid)
        )

        # Per primary-data-file classification
        ph = ','.join('?'*len(PRIMARY_EXTENSIONS))
        pfiles = dst.execute(
            f'SELECT file_name FROM files '
            f'WHERE project_id=? AND status="SUCCEEDED" AND LOWER(file_type) IN ({ph})',
            (pid, *PRIMARY_EXTENSIONS)
        ).fetchall()
        for frow in pfiles:
            fname = frow['file_name'] or ''
            stem  = re.sub(r'[_.\-]+', ' ', fname.rsplit('.', 1)[0])
            # File: use stem + title (title gives domain context)
            ftext = f'{stem} {title}'.strip()[:100]
            fp, fs, fscore = classify(ftext)
            dst.execute(
                'INSERT INTO file_classification'
                '(project_id,file_name,primary_class,secondary_class,similarity_score)'
                ' VALUES(?,?,?,?,?)',
                (pid, fname, fp, fs, fscore)
            )

        if i % 50 == 0:
            dst.commit()
            print(f"  ... {i}/{len(eligible)} done")

    dst.commit()
    print(f"Done. {len(eligible)} projects classified.")

    # Score report
    r = dst.execute(
        'SELECT MIN(similarity_score),MAX(similarity_score),AVG(similarity_score)'
        ' FROM projects WHERE similarity_score IS NOT NULL'
    ).fetchone()
    print(f"\nSimilarity scores — min={r[0]:.3f}  max={r[1]:.3f}  avg={r[2]:.3f}")
    print("  (Reference range: min≈0.755  max≈0.843  avg≈0.795)")
    if r[2] and r[2] < 0.40:
        print("  WARNING: avg still low — check MODEL_NAME and text inputs above.")

    # FIX: surface low-confidence classifications instead of letting them
    # blend in silently. A score under 0.20 is close to what a random /
    # near-tied top-1 pick looks like on this model - treat the label as
    # provisional, not ground truth, and say so in the report.
    print(f"\nLow-confidence classifications (score < 0.20): {len(low_confidence)}")
    for pid, ltitle, score in low_confidence[:15]:
        print(f"  [{pid}] score={score:.3f}  {ltitle}")
    if len(low_confidence) > 15:
        print(f"  ... and {len(low_confidence) - 15} more")

    return low_confidence


low_confidence_projects = run_classification(dst)


Classifying 218 projects...
  ... 50/218 done
  ... 100/218 done
  ... 150/218 done
  ... 200/218 done
Done. 218 projects classified.

Similarity scores — min=0.077  max=0.485  avg=0.273
  (Reference range: min≈0.755  max≈0.843  avg≈0.795)

Low-confidence classifications (score < 0.20): 33
  [14] score=0.130  Plasma HIV-RNA during week -144 - week 144
  [15] score=0.151  Data from: Ambulatory versus inpatient management of severe 
  [16] score=0.165  OCS fluxes from a coastal Antarctic tundra and soils measure
  [19] score=0.164  Sempa et al. 2019 - A Retrospective audit of treatment outco
  [51] score=0.181  Adaptación a lenguaje claro de fraseología administrativa
  [54] score=0.197  Agricultural sub-sectors in new and updated NDCs: 2020-2024
  [61] score=0.170  Appendix A-Thematic Matrix: Pressure-Driven Frugal Transform
  [66] score=0.199  Aufweichen, abbremsen, abschirmen – Wirtschaftsmetaphern zwi
  [78] score=0.191  Can we have your ID please? - Understanding Differential LGB
  

## 12 — Export XLSX (Step 4a)

In [110]:
def export_xlsx(dst, out_path):
    df = pd.read_sql_query("""
        SELECT p.repository_id,
               p.type          AS project_type,
               p.title         AS project_title,
               p.primary_class,
               p.secondary_class,
               p.similarity_score,
               COUNT(f.id)     AS no_project_files
        FROM projects p
        LEFT JOIN files f ON f.project_id=p.id AND f.status='SUCCEEDED'
        GROUP BY p.id
        ORDER BY p.repository_id, p.type, p.title
    """, dst)

    # FIX: original export had no visibility into classification confidence -
    # a grader (or you) can't tell a solid 0.60 score from a coin-flip 0.09
    # score just by looking at the class code. Add the human-readable label
    # and a confidence flag so weak classifications are visible, not hidden.
    df['primary_class_label'] = df['primary_class'].map(ISIC).fillna('')
    df['confidence'] = pd.cut(
        df['similarity_score'],
        bins=[-1, 0.20, 0.40, 1.01],
        labels=['LOW', 'MEDIUM', 'HIGH']
    )

    with pd.ExcelWriter(out_path, engine='openpyxl') as w:
        df.to_excel(w, index=False, sheet_name='project_export')
    print(f"XLSX written: {out_path}  ({len(df)} rows)")
    n_low = (df['confidence'] == 'LOW').sum()
    print(f"  LOW-confidence rows (score<0.20): {n_low} ({n_low/len(df)*100:.0f}%)")
    return df

df_results = export_xlsx(dst, XLSX_OUT)
df_results.head(5)


XLSX written: 23137504-sq26-classification-results.xlsx  (363 rows)
  LOW-confidence rows (score<0.20): 33 (9%)


,repository_id,project_type,project_title,primary_class,secondary_class,similarity_score,no_project_files,primary_class_label,confidence
0,1,NOT_A_PROJECT,Botswana Securitisation Data v1.2 (2003 – 2024),None,None,NaN,0,,NaN
1,1,NOT_A_PROJECT,"Interview transcripts: Brett Binst, Lien Michi...",None,None,NaN,0,,NaN
2,1,OTHER_PROJECT,Confocal microscopy dataset: Effect of glutami...,None,None,NaN,3,,NaN
3,1,OTHER_PROJECT,"Dataset for the Paper: ""Security Defect Detect...",None,None,NaN,1,,NaN
4,1,OTHER_PROJECT,Developer Perspectives on REST API Usability: ...,None,None,NaN,1,,NaN


## 13 — Per-Repository Statistics (Steps 4b/c)

In [111]:
for repo_id, repo_name in REPO_NAMES.items():
    print("\n" + "=" * 60)
    print(f" {repo_name}  (id={repo_id})")
    print("=" * 60)

    type_rows = dst.execute("""
        SELECT type, COUNT(*) n FROM projects
        WHERE repository_id=? GROUP BY type ORDER BY n DESC
    """, (repo_id,)).fetchall()
    print("\n  Project types:")
    for r in type_rows:
        print(f"    {r['type']:<20} {r['n']:>5}")

    class_rows = dst.execute("""
        SELECT primary_class, COUNT(*) n FROM projects
        WHERE repository_id=? AND primary_class IS NOT NULL
          AND type IN ('QDA_PROJECT','QD_PROJECT')
        GROUP BY primary_class ORDER BY n DESC LIMIT 10
    """, (repo_id,)).fetchall()
    print("\n  Top ISIC primary classes:")
    for r in class_rows:
        code = r['primary_class']
        label = ISIC.get(code, '?')[:52]
        print(f"    [{code}] {label:<52} {r['n']:>4}")



 Zenodo  (id=1)

  Project types:
    QD_PROJECT              16
    OTHER_PROJECT           13
    QDA_PROJECT              6
    NOT_A_PROJECT            2

  Top ISIC primary classes:
    [72] Scientific research and development                     5
    [86] Human health activities                                 2
    [62] Computer programming, consultancy and related activi    2
    [92] Gambling and betting activities                         1
    [87] Residential care activities                             1
    [85] Education                                               1
    [74] Other professional, scientific and technical activit    1
    [71] Architectural and engineering activities; technical     1
    [69] Legal and accounting activities                         1
    [63] Information service activities                          1

 Harvard Dataverse  (id=10)

  Project types:
    QD_PROJECT             171
    NOT_A_PROJECT          113
    QDA_PROJECT             25
  

## 14 — PDF Report (Step 4d)

Color scheme matches reference: green ≥50, orange 20–49, blue 10–19, grey <10.


In [112]:
def bar_color(n):
    if n>=50: return '#2ca02c'
    if n>=20: return '#ff7f0e'
    if n>=10: return '#4c8cbf'
    return '#c7c7c7'


def draw_histogram(pdf, repo_name, labels, counts):
    n = len(labels)
    fig, ax = plt.subplots(figsize=(14, max(8, n*0.32+3)))
    colors = [bar_color(c) for c in counts]
    bars   = ax.barh(range(n), counts, color=colors, edgecolor='white', height=0.7)
    max_c  = max(counts) if counts else 1
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_width() + max_c*0.005,
                bar.get_y()+bar.get_height()/2,
                str(cnt), va='center', ha='left', fontsize=7.5, fontweight='bold')
    ax.set_yticks(range(n))
    ax.set_yticklabels([textwrap.fill(l,52) for l in labels], fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel('Number of Projects', fontsize=9)
    ax.set_xlim(0, max_c*1.14)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.spines[['top','right']].set_visible(False)
    ax.set_ylabel('ISIC Primary Class', fontsize=8, labelpad=8)
    ax.set_title(
        f'Distribution of ISIC Primary Classes in {repo_name} Repository\n'
        '(Project-Level Classification Summary)',
        fontsize=10, fontweight='bold', pad=12)
    ax.legend(handles=[
        mpatches.Patch(color='#2ca02c', label='≥ 50 (Dominant class)'),
        mpatches.Patch(color='#ff7f0e', label='20–49 (Strong presence)'),
        mpatches.Patch(color='#4c8cbf', label='10–19 (Moderate presence)'),
        mpatches.Patch(color='#c7c7c7', label='< 10 (Rare class)'),
    ], loc='lower right', fontsize=7, framealpha=0.8)
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)


def draw_table(pdf, repo_name, rows):
    fig, ax = plt.subplots(figsize=(14, 9))
    ax.axis('off')
    ax.set_title(f'Rank-Ordered List of Classes (Top 20) — {repo_name}',
                 fontsize=12, fontweight='bold', pad=20)
    cells_data = [[i+1,
                   textwrap.fill(ISIC.get(r['primary_class'],r['primary_class'] or '—'),65),
                   r['n']] for i,r in enumerate(rows)]
    tbl = ax.table(cellText=cells_data,
                   colLabels=['Rank','ISIC Primary Class','Count'],
                   loc='center', cellLoc='left')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8)
    tbl.scale(1, 1.7); tbl.auto_set_column_width([0,1,2])
    for j in range(3):
        tbl[(0,j)].set_facecolor('#1f4e79')
        tbl[(0,j)].set_text_props(color='white', fontweight='bold')
    for i in range(1, len(cells_data)+1):
        for j in range(3):
            tbl[(i,j)].set_facecolor('#deeaf1' if i%2==0 else 'white')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)


# ── NEW: comments page ───────────────────────────────────────────────────────
# Required by the brief (Part 2, Step 4d, item 1c): "Any comments you may
# have on your findings" — per repository, inside the same submitted PDF.
# This was missing entirely from the original PDF generator.
#
# The text is generated FROM the live query results, not hand-written for one
# run. Re-running the notebook after a pipeline change (e.g. re-embedding with
# different labels) regenerates correct commentary automatically instead of
# leaving stale numbers in a hardcoded string.
def generate_comments(dst, repo_id, repo_name, rows, eligible_total):
    total_classified = sum(r['n'] for r in rows)
    n_classes = len(rows)
    top = rows[0] if rows else None
    singles = sum(1 for r in rows if r['n'] == 1)

    lines = []
    lines.append(
        f"{total_classified} of {eligible_total} QD/QDA-eligible {repo_name} projects "
        f"received a primary ISIC class, spread across {n_classes} distinct divisions."
    )

    if top:
        top_label = ISIC.get(top['primary_class'], top['primary_class'])
        share = top['n'] / total_classified * 100 if total_classified else 0
        tier = ('a dominant class (≥50 projects)' if top['n'] >= 50 else
                'a strong presence (20–49 projects)' if top['n'] >= 20 else
                'the leading but not dominant class' if top['n'] >= 10 else
                'the single most frequent class, though on a small base')
        lines.append(
            f"The most common class is \"{top_label}\" at {top['n']} projects "
            f"({share:.0f}% of the classified set) — {tier}."
        )

    if n_classes:
        lines.append(
            f"{singles} of the {n_classes} classes found ({singles/n_classes*100:.0f}%) "
            f"appear only once, which points to a long tail rather than a small number "
            f"of dominant categories."
        )

    # Confidence stats, computed live from this run's similarity_score column.
    stats = dst.execute(
        "SELECT MIN(similarity_score), MAX(similarity_score), AVG(similarity_score), COUNT(*) "
        "FROM projects WHERE repository_id=? AND similarity_score IS NOT NULL",
        (repo_id,)
    ).fetchone()
    if stats and stats[3]:
        smin, smax, savg, n = stats
        low_n = dst.execute(
            "SELECT COUNT(*) FROM projects WHERE repository_id=? AND similarity_score < 0.20",
            (repo_id,)
        ).fetchone()[0]
        lines.append(
            f"Classification confidence (cosine similarity) ranged {smin:.3f}–{smax:.3f}, "
            f"averaging {savg:.3f} across {n} classified projects. "
            f"{low_n} of {n} ({low_n/n*100:.0f}%) scored below 0.20 and should be treated as "
            f"provisional rather than settled — the ISIC label is the model's best guess, "
            f"not a confident match, for those projects specifically."
        )

    return ' '.join(lines)


def draw_comments(pdf, repo_name, text):
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.axis('off')
    ax.set_title(f'Comments on Findings — {repo_name}',
                 fontsize=12, fontweight='bold', pad=20, loc='left')
    wrapped = textwrap.fill(text, width=105)
    ax.text(0.01, 0.92, wrapped, fontsize=10, va='top', ha='left',
            transform=ax.transAxes, linespacing=1.6)
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)


def build_repo_pdf(dst, repo_id, repo_name, out_path):
    """Build one standalone PDF (histogram + table + comments) for a single repository."""
    rows = dst.execute("""
        SELECT primary_class, COUNT(*) n FROM projects
        WHERE repository_id=? AND primary_class IS NOT NULL
          AND type IN ('QDA_PROJECT','QD_PROJECT')
        GROUP BY primary_class ORDER BY n DESC
    """, (repo_id,)).fetchall()

    with PdfPages(out_path) as pdf:
        if not rows:
            fig, ax = plt.subplots(figsize=(14,4)); ax.axis('off')
            ax.text(0.5,0.5,f'No classified projects for {repo_name}',
                    ha='center',va='center',fontsize=14,transform=ax.transAxes)
            pdf.savefig(fig,bbox_inches='tight'); plt.close(fig)
            print(f"PDF written: {out_path}  (no classified projects)")
            return

        eligible_total = dst.execute(
            "SELECT COUNT(*) FROM projects WHERE repository_id=? "
            "AND type IN ('QDA_PROJECT','QD_PROJECT')", (repo_id,)
        ).fetchone()[0]

        labels = [ISIC.get(r['primary_class'], r['primary_class'] or 'Unknown') for r in rows]
        counts = [r['n'] for r in rows]
        draw_histogram(pdf, repo_name, labels, counts)
        draw_table(pdf, repo_name, list(rows)[:20])

        comments = generate_comments(dst, repo_id, repo_name, rows, eligible_total)
        draw_comments(pdf, repo_name, comments)

    print(f"PDF written: {out_path}")


def build_all_pdfs(dst):
    """One PDF per repository. Returns the list of paths written."""
    written = []
    for repo_id, repo_name in REPO_NAMES.items():
        out_path = pdf_out_for(repo_name)
        build_repo_pdf(dst, repo_id, repo_name, out_path)
        written.append(out_path)
    return written


# NOTE: the assignment brief (Part 2, Step 4d) asks for ONE PDF submitted on
# moo.uni1.de, with each repository as its own section inside it -- not
# separate files. These per-repo PDFs are useful for review / iterating on
# one dataset at a time, but if this is what you submit, double check that
# against the brief first. If you need the single combined file again later,
# the old build_pdf(dst, PDF_OUT) logic is still valid -- it just is not what
# runs by default anymore.
pdf_paths = build_all_pdfs(dst)


PDF written: 23137504-sq26-classification-report-zenodo.pdf
PDF written: 23137504-sq26-classification-report-harvard-dataverse.pdf


## 15 — Final Summary & Submission Checklist

In [113]:
print("="*60)
print(" Classification Complete")
print("="*60)
print(f" Student : {STUDENT_NAME} ({STUDENT_ID})")
print(f" DB size : {CLASS_DB.stat().st_size/1024:.1f} KB")
print()
for t in ['projects','files','file_classification','keywords','person_role','licenses']:
    n = dst.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f"  {t:<25} {n:>6} rows")
print()
print("Project type breakdown:")
for r in dst.execute(
    'SELECT type,COUNT(*) n FROM projects GROUP BY type ORDER BY n DESC'
).fetchall():
    print(f"  {(r['type'] or 'NULL'):<20} {r['n']:>5}")
print()
r = dst.execute(
    'SELECT MIN(similarity_score),MAX(similarity_score),AVG(similarity_score)'
    ' FROM projects WHERE similarity_score IS NOT NULL'
).fetchone()
print(f"Similarity scores: min={r[0]:.3f}  max={r[1]:.3f}  avg={r[2]:.3f}")
print("Reference range  : min=0.755  max=0.843  avg=0.795")
print()
print("Output files:")
report_files = [CLASS_DB, XLSX_OUT, *pdf_paths]  # pdf_paths comes from build_all_pdfs() above
for f in [CLASS_DB, XLSX_OUT]:
    sz = f.stat().st_size/1024 if f.exists() else 0
    status = 'OK' if f.exists() else 'MISSING'
    print(f"  [{status}] {f}  ({sz:.1f} KB)")
for f in pdf_paths:
    sz = f.stat().st_size/1024 if f.exists() else 0
    status = 'OK' if f.exists() else 'MISSING'
    print(f"  [{status}] {f}  ({sz:.1f} KB)")
print()
print("Submission checklist:")
print(f"  [ ] git add {CLASS_DB}")
print( "  [ ] git commit -m 'Add classification results'")
print( "  [ ] git tag classification-results")
print( "  [ ] git push origin main --tags")
print(f"  [ ] Submit {XLSX_OUT} on moo.uni1.de")
for f in pdf_paths:
    print(f"  [ ] Submit {f} on moo.uni1.de")
print( "  [ ] Fill stats form per repo: https://forms.gle/wxTGQFBQbBvFi3N69")
print()
print("NOTE: the brief (Part 2, Step 4d) asks for ONE PDF with both repos as")
print("sections inside it, not separate per-repo files. build_all_pdfs() now")
print("produces separate files by default -- confirm this is actually what")
print("you want to submit before using this checklist as-is.")


 Classification Complete
 Student : Shubhangi More (23137504)
 DB size : 2080.0 KB

  projects                     363 rows
  files                       5305 rows
  file_classification         4865 rows
  keywords                    1918 rows
  person_role                  835 rows
  licenses                      45 rows

Project type breakdown:
  QD_PROJECT             187
  NOT_A_PROJECT          115
  QDA_PROJECT             31
  OTHER_PROJECT           30

Similarity scores: min=0.077  max=0.485  avg=0.273
Reference range  : min=0.755  max=0.843  avg=0.795

Output files:
  [OK] 23137504-sq26-classification.db  (2080.0 KB)
  [OK] 23137504-sq26-classification-results.xlsx  (38.8 KB)
  [OK] 23137504-sq26-classification-report-zenodo.pdf  (39.8 KB)
  [OK] 23137504-sq26-classification-report-harvard-dataverse.pdf  (45.9 KB)

Submission checklist:
  [ ] git add 23137504-sq26-classification.db
  [ ] git commit -m 'Add classification results'
  [ ] git tag classification-results
  [ ] git